In [ ]:
!pip install segmentation-models-pytorch timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

# Mount Google Drive (run this once per session)
drive.mount('/content/drive')

# After mounting, your Drive appears here:
print("Your Drive is mounted at: /content/drive/MyDrive/")

Mounted at /content/drive
Your Drive is mounted at: /content/drive/MyDrive/


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
class DefectDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        # Load as grayscale
        image = np.array(Image.open(img_path).convert("L"))
        mask = np.array(Image.open(mask_path).convert("L"))

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Convert mask to binary float (0 or 1), add channel dimension
        mask = (mask > 127).float().unsqueeze(0)  # 1xHxW

        return image, mask

In [ ]:
# Your dataset paths (fixed the double slash typo)
image_dir = '/content/drive/MyDrive/images-unet-train'
mask_dir  = '/content/drive/MyDrive/images-unet-train/image-unet-train-binary'

# Where to save the best model (recommended: put it in Drive)
BEST_MODEL_PATH = '/content/drive/MyDrive/images-unet-train/best_defect_unet_ROI_resnet34.pth'

# Optional: print to confirm paths exist
print("Image dir exists?", os.path.isdir(image_dir))
print("Mask dir exists?",  os.path.isdir(mask_dir))

Image dir exists? True
Mask dir exists? True


In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.ElasticTransform(alpha=1, sigma=50, p=0.5),      # Fixed: removed deprecated param
    A.GridDistortion(p=0.5),
    A.Resize(height=256, width=512),                   # Good for both small & wide images
    A.Normalize(mean=0.5, std=0.5),                    # For single-channel grayscale
    ToTensorV2(),                                      # Converts to torch Tensor (1xHxW)
])

val_transform = A.Compose([
    A.Resize(height=256, width=512),
    A.Normalize(mean=0.5, std=0.5),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
# Train on ALL images with augmentation
train_dataset = DefectDataset(image_dir, mask_dir, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True)

# Validation: same images but without heavy augmentation (for monitoring)
val_dataset = DefectDataset(image_dir, mask_dir, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f"Dataset loaded: {len(train_dataset)} images")

Dataset loaded: 227 images


In [ ]:
model = smp.Unet(
    encoder_name="resnet34",        # Pretrained encoder (adapts to grayscale automatically)
    encoder_weights="imagenet",
    in_channels=1,                  # Grayscale input
    classes=1,                      # Binary segmentation
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")

Using device: cuda


In [ ]:
criterion = smp.losses.DiceLoss(mode='binary')  # Great for small/imbalanced defects
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# ────────────────────────────────────────────────
#               Training Loop – Save to Drive
# ────────────────────────────────────────────────

num_epochs = 100
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ---------- Training ----------
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ---------- Validation ----------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Save best model → to Google Drive
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"    → New best model saved to Drive: {BEST_MODEL_PATH}")

print("\nTraining completed!")
print(f"Best model saved as: {BEST_MODEL_PATH}")
print(f"Best validation loss achieved: {best_val_loss:.4f}")

Epoch 1/100 [Train]: 100%|██████████| 113/113 [03:26<00:00,  1.83s/it]


Epoch   1 | Train Loss: 0.6450 | Val Loss: 0.5930
    → New best model saved to Drive: /content/drive/MyDrive/images-unet-train/best_defect_unet_ROI_resnet34.pth


Epoch 2/100 [Train]: 100%|██████████| 113/113 [00:11<00:00,  9.48it/s]


Epoch   2 | Train Loss: 0.5168 | Val Loss: 0.4887
    → New best model saved to Drive: /content/drive/MyDrive/images-unet-train/best_defect_unet_ROI_resnet34.pth


Epoch 3/100 [Train]: 100%|██████████| 113/113 [00:13<00:00,  8.26it/s]


...

Epoch  98 | Train Loss: 0.1825 | Val Loss: 0.3169


Epoch 99/100 [Train]: 100%|██████████| 113/113 [00:10<00:00, 10.51it/s]


Epoch  99 | Train Loss: 0.2021 | Val Loss: 0.2778


Epoch 100/100 [Train]: 100%|██████████| 113/113 [00:11<00:00, 10.20it/s]


Epoch 100 | Train Loss: 0.2047 | Val Loss: 0.2610

Training completed!
Best model saved as: /content/drive/MyDrive/images-unet-train/best_defect_unet_ROI_resnet34.pth
Best validation loss achieved: 0.2379
